# Book Genre Classification with LSTM

This notebook trains a professional LSTM-based deep learning model for book genre classification using the high-confidence 22-class dataset `BooksClassifier_dataset_high_confidence_22.csv`.

The file extension is `.xls`, but the dataset content is CSV. The important columns are:

- `model_text`: input text, combining cleaned title and cleaned description
- `target_genre`: target class label
- `title_clean`: cleaned book title

Expected accuracy after tuning is usually around **70% to 82%** for this 22-class dataset. A result above 75% is good, and 80%+ is very strong for an LSTM model on this task.

## 1. Install Dependencies

Run this cell only if the packages are missing in your environment.

In [ ]:
# Uncomment if needed:
# !pip install pandas numpy scikit-learn matplotlib seaborn tensorflow

## 2. Imports and Configuration

In [ ]:
from pathlib import Path
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_PATH = Path("BooksClassifier_dataset_high_confidence_22.csv")
# For much higher accuracy, if broad categories are allowed, use:
# DATASET_PATH = Path("BooksClassifier_dataset_reduced_12class.csv")
OUTPUT_DIR = Path("outputs/keras_lstm_books_classifier")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "model_text"
LABEL_COLUMN = "target_genre"
TITLE_COLUMN = "title_clean"

INCLUDE_TITLE = False
MAX_TOKENS = 50000
MAX_LENGTH = 320
EMBEDDING_DIM = 256
LSTM_UNITS = 192
DROPOUT = 0.45

LEARNING_RATES = [5e-4, 3e-4, 1e-4]
BATCH_SIZES = [32, 64]
CLASS_WEIGHT_MODES = ["none", "sqrt", "balanced"]
EPOCHS = 30
PATIENCE = 5
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nFull-row duplicates:", df.duplicated().sum())
print("Duplicate cleaned_text rows:", df.duplicated(subset=[TEXT_COLUMN]).sum())

print("\nClass distribution:")
class_counts = df[LABEL_COLUMN].value_counts()
print(class_counts)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=class_counts.values, y=class_counts.index, palette="viridis")
plt.title("Class Distribution")
plt.xlabel("Number of samples")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

word_lengths = df[TEXT_COLUMN].astype(str).str.split().str.len()
print(word_lengths.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

plt.figure(figsize=(10, 4))
sns.histplot(word_lengths.clip(upper=500), bins=50)
plt.title("Text Length Distribution, clipped at 500 words")
plt.xlabel("Words")
plt.tight_layout()
plt.show()

## 4. Clean and Prepare Data

The dataset already contains a cleaned semantic text column, so preprocessing is intentionally conservative. We normalize spacing and case, remove empty rows if any, and remove duplicate text-label pairs to reduce leakage.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("\u00a0", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value.casefold()


data = df.copy()
data[TEXT_COLUMN] = data[TEXT_COLUMN].map(normalize_text)
data[LABEL_COLUMN] = data[LABEL_COLUMN].astype(str).str.strip()

if INCLUDE_TITLE and TITLE_COLUMN in data.columns:
    data[TITLE_COLUMN] = data[TITLE_COLUMN].map(normalize_text)
    data[TEXT_COLUMN] = (data[TITLE_COLUMN] + " " + data[TEXT_COLUMN]).str.strip()

data = data.replace({TEXT_COLUMN: {"": np.nan}, LABEL_COLUMN: {"": np.nan}})
data = data.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
data = data.drop_duplicates()
data = data.drop_duplicates(subset=[TEXT_COLUMN, LABEL_COLUMN])

conflicting_texts = data.groupby(TEXT_COLUMN)[LABEL_COLUMN].nunique()
conflicting_texts = conflicting_texts[conflicting_texts > 1].index
if len(conflicting_texts) > 0:
    data = data[~data[TEXT_COLUMN].isin(conflicting_texts)]

data = data.reset_index(drop=True)

print("Final shape:", data.shape)
print("Final class count:", data[LABEL_COLUMN].nunique())
print(data[LABEL_COLUMN].value_counts())

## 5. Encode Labels and Split Train / Validation / Test

In [ ]:
label_encoder = LabelEncoder()
data["label_id"] = label_encoder.fit_transform(data[LABEL_COLUMN])
class_names = list(label_encoder.classes_)
num_classes = len(class_names)

train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label_id"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_id"],
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Classes:", num_classes)

with open(OUTPUT_DIR / "label_classes.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)

## 6. Text Vectorization and TensorFlow Datasets

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LENGTH,
    standardize=None,
    split="whitespace",
)

vectorizer.adapt(train_df[TEXT_COLUMN].values)
vocab_size = len(vectorizer.get_vocabulary())
print("Vocabulary size:", vocab_size)

def make_dataset(frame, batch_size, shuffle=False):
    texts = frame[TEXT_COLUMN].astype(str).values
    labels = frame["label_id"].astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

balanced_class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_df["label_id"].values,
)
sqrt_class_weights_array = np.sqrt(balanced_class_weights_array)

class_weight_options = {
    "none": None,
    "sqrt": {i: float(weight) for i, weight in enumerate(sqrt_class_weights_array)},
    "balanced": {i: float(weight) for i, weight in enumerate(balanced_class_weights_array)},
}

pd.DataFrame({
    "genre": class_names,
    "balanced_weight": balanced_class_weights_array,
    "sqrt_weight": sqrt_class_weights_array,
}).sort_values("balanced_weight", ascending=False).head(10)

## 7. Build the LSTM Model

This architecture uses an embedding layer, stacked bidirectional LSTMs, attention pooling, dropout, weight decay, and gradient clipping. It is designed to capture ordered context in book descriptions while remaining efficient enough for a medium-sized dataset.

In [ ]:
@tf.keras.utils.register_keras_serializable()
class AttentionPooling(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True

    def build(self, input_shape):
        self.score = tf.keras.layers.Dense(1)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        scores = tf.squeeze(self.score(inputs), axis=-1)
        if mask is not None:
            mask = tf.cast(mask, scores.dtype)
            scores = scores + (1.0 - mask) * tf.constant(-1e9, dtype=scores.dtype)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * tf.expand_dims(weights, axis=-1), axis=1)

    def compute_mask(self, inputs, mask=None):
        return None


def build_lstm_model(learning_rate):
    text_input = tf.keras.Input(shape=(), dtype=tf.string, name="book_text")
    x = vectorizer(text_input)
    x = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        name="token_embedding",
    )(x)
    x = tf.keras.layers.SpatialDropout1D(DROPOUT)(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=0.20),
        name="bilstm_1",
    )(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS // 2, return_sequences=True, dropout=0.20),
        name="bilstm_2",
    )(x)
    attention_pool = AttentionPooling(name="attention_pool")(x)
    max_pool = tf.keras.layers.GlobalMaxPooling1D(name="max_pool")(x)
    x = tf.keras.layers.Concatenate()([attention_pool, max_pool])
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    x = tf.keras.layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY))(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    output = tf.keras.layers.Dense(num_classes, activation="softmax", name="genre")(x)

    model = tf.keras.Model(text_input, output, name="books_bilstm_genre_classifier")
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=WEIGHT_DECAY,
        clipnorm=GRADIENT_CLIP_NORM,
    )
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


preview_model = build_lstm_model(LEARNING_RATES[0])
preview_model.summary()

## 8. Hyperparameter Search with Early Stopping

In [ ]:
search_results = []
best_val_accuracy = -1.0
best_model_path = None

for learning_rate in LEARNING_RATES:
    for batch_size in BATCH_SIZES:
        for class_weight_mode in CLASS_WEIGHT_MODES:
            current_class_weight = class_weight_options[class_weight_mode]
            print("\n" + "=" * 80)
            print(
                f"Training candidate: learning_rate={learning_rate}, "
                f"batch_size={batch_size}, class_weight_mode={class_weight_mode}"
            )
            print("=" * 80)

            train_ds = make_dataset(train_df, batch_size=batch_size, shuffle=True)
            val_ds = make_dataset(val_df, batch_size=batch_size, shuffle=False)

            model = build_lstm_model(learning_rate)
            candidate_path = OUTPUT_DIR / f"best_lstm_lr{learning_rate:g}_bs{batch_size}_cw{class_weight_mode}.keras"

            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_accuracy",
                    mode="max",
                    patience=PATIENCE,
                    restore_best_weights=True,
                    verbose=1,
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor="val_accuracy",
                    mode="max",
                    factor=0.5,
                    patience=2,
                    min_lr=1e-6,
                    verbose=1,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    filepath=candidate_path,
                    monitor="val_accuracy",
                    mode="max",
                    save_best_only=True,
                    verbose=1,
                ),
            ]

            start = time.time()
            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=EPOCHS,
                class_weight=current_class_weight,
                callbacks=callbacks,
                verbose=1,
            )

            candidate_best = max(history.history["val_accuracy"])
            result = {
                "learning_rate": learning_rate,
                "batch_size": batch_size,
                "class_weight_mode": class_weight_mode,
                "best_val_accuracy": float(candidate_best),
                "epochs_ran": len(history.history["loss"]),
                "seconds": time.time() - start,
                "model_path": str(candidate_path),
            }
            search_results.append(result)

            if candidate_best > best_val_accuracy:
                best_val_accuracy = candidate_best
                best_model_path = candidate_path

pd.DataFrame(search_results).sort_values("best_val_accuracy", ascending=False)

## 9. Evaluate on the Test Set

In [ ]:
print("Best model:", best_model_path)
best_model = tf.keras.models.load_model(best_model_path, custom_objects={"AttentionPooling": AttentionPooling})

test_ds = make_dataset(test_df, batch_size=64, shuffle=False)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)

y_true = test_df["label_id"].values
y_proba = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_proba, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0,
)

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_weighted": float(precision),
    "recall_weighted": float(recall),
    "f1_weighted": float(f1),
    "test_loss": float(test_loss),
}

print(metrics)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=False,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(OUTPUT_DIR / "confusion_matrix.csv")
with open(OUTPUT_DIR / "final_results.json", "w", encoding="utf-8") as f:
    json.dump({"metrics": metrics, "search_results": search_results, "best_model_path": str(best_model_path)}, f, indent=2)

print("Saved results to:", OUTPUT_DIR)
print("Saved best model to:", best_model_path)

## 10. Predict a New Book Description

In [ ]:
def predict_genre(text, top_k=5):
    text = normalize_text(text)
    input_text = tf.constant([text], dtype=tf.string)
    probabilities = best_model.predict(input_text, verbose=0)[0]
    top_indices = np.argsort(probabilities)[::-1][:top_k]
    return pd.DataFrame({
        "genre": [class_names[i] for i in top_indices],
        "probability": [float(probabilities[i]) for i in top_indices],
    })


predict_genre("A detective investigates a strange murder in a quiet town while hidden secrets begin to appear.")